<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-01-setup-and-iam/lesson-1.1-setup/notebooks/GCP_Capstone_1.1_Setup.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1.1 Setting Up Your GCP AI Project
**Netsetos GenAI Engineering — GCP Capstone**

This notebook verifies your GCP project setup and runs your first Gemini calls.


## Step 1: Install Dependencies


In [ ]:
!pip install -q google-genai google-cloud-aiplatform google-cloud-firestore google-cloud-storage


## Step 2: Authenticate (if running on Colab)


In [ ]:
# Only needed on Google Colab (not Cloud Shell or Workbench)
from google.colab import auth
auth.authenticate_user()


## Step 3: Set Your Project


In [ ]:
PROJECT_ID = "documind-ai-YOUR-ID"  # <-- CHANGE THIS
LOCATION = "us-central1"

!gcloud config set project {PROJECT_ID}
print(f"Project: {PROJECT_ID}")


## Step 4: Verify APIs are Enabled


In [ ]:
!gcloud services list --enabled --format="table(name)" | head -20

# Count enabled APIs
import subprocess
result = subprocess.run(["gcloud", "services", "list", "--enabled", "--format=value(name)"], capture_output=True, text=True)
apis = result.stdout.strip().split("\n")
print(f"\n✅ {len(apis)} APIs enabled")


## Step 5: Your First Gemini Call


In [ ]:
from google import genai

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain what Google Cloud Platform is in exactly 2 sentences."
)

print("✅ Gemini says:")
print(response.text)

usage = response.usage_metadata
print(f"\n📊 Token Usage:")
print(f"  Input tokens:  {usage.prompt_token_count}")
print(f"  Output tokens: {usage.candidates_token_count}")
print(f"  Total tokens:  {usage.total_token_count}")

cost = (usage.prompt_token_count * 0.15 + usage.candidates_token_count * 0.60) / 1_000_000
print(f"  Cost: ${cost:.6f} (₹{cost*93:.4f})")


## Step 6: Token Counting (Free — No Inference Cost)


In [ ]:
# count_tokens is FREE — no Gemini inference happens
texts = [
    "Hello",
    "Hyderabad is a beautiful city in India",
    "The transformer architecture uses self-attention mechanisms to process sequences in parallel",
    "Explain the complete architecture of a retrieval-augmented generation system with vector databases" * 10,
]

for text in texts:
    result = client.models.count_tokens(
        model="gemini-2.5-flash",
        contents=text
    )
    cost_input = result.total_tokens * 0.15 / 1_000_000
    pct = result.total_tokens / 1_000_000 * 100
    print(f"  {result.total_tokens:>6,} tokens | ${cost_input:.6f} | {pct:.4f}% of 1M context | {text[:50]}...")


## Step 7: Compare 3 Gemini Models


In [ ]:
import time

prompt = "Explain what a Large Language Model is in 3 sentences."
models = ["gemini-2.5-flash-lite-preview-06-17", "gemini-2.5-flash", "gemini-2.5-pro"]

for model_name in models:
    start = time.time()
    try:
        resp = client.models.generate_content(model=model_name, contents=prompt)
        latency = (time.time() - start) * 1000
        u = resp.usage_metadata
        cost = (u.prompt_token_count * 0.15 + u.candidates_token_count * 0.60) / 1_000_000
        print(f"\n📌 {model_name}")
        print(f"   Tokens: {u.total_token_count} | Latency: {latency:.0f}ms | Cost: ${cost:.6f}")
        print(f"   Response: {resp.text[:150]}...")
    except Exception as e:
        print(f"\n❌ {model_name}: {e}")


## Step 8: Budget Check


In [ ]:
# Check your remaining credits
print("To check credits: GCP Console → Billing → Overview → Credit Details")
print("Or run: gcloud billing projects describe " + PROJECT_ID)


## ✅ Setup Complete!

Your GCP GenAI environment is verified and ready:
- ✅ Project created and configured
- ✅ Billing linked with budget alerts
- ✅ 15+ APIs enabled
- ✅ google-genai SDK installed
- ✅ Gemini API responding
- ✅ Token counting working
- ✅ Model comparison complete

**Next: Lesson 1.2 — IAM & Security**
